# 多头注意力（Multi-Head Attention）

> Transformer 的核心组件，让模型在不同子空间中关注不同位置的信息。

## 背景
单头注意力只有一个注意力分布，表达能力有限。多头注意力将 Q/K/V 投影到多个子空间，
各自独立做 attention 后拼接，再通过输出投影恢复原维度。每个 head 可以关注不同模式
（如语法、语义、位置等），大幅提升表达能力。

## 公式
$$\text{MHA}(Q,K,V) = \text{Concat}(head_1, ..., head_h) W^O$$
$$head_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

## 复杂度
- 时间：O(n² × d)，n=序列长度，d=模型维度
- 空间：O(n² × h)，h=head 数量（attention 矩阵）
- 参数：4 × d²（QKV + Output 投影）

## 考察点
- 为什么要除以 √d_k：防止点积过大导致 softmax 饱和
- head 数量选择：太多则每个 head 维度太小，太少则表达力不足
- 与 GQA/MQA 的关系：共享 KV head 减少显存


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LlamaMHA(nn.Module):
    def __init__(self, dim, num_heads, head_dim=None):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim if head_dim is not None else dim // num_heads
        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x, attention_mask=None, cache=None):
        B, N, _ = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)
        attn = (q @ k.transpose(-1, -2)) * self.scale
        if attention_mask is not None:
            attn = attn + attention_mask
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, N, -1)
        return self.o_proj(out), new_cache

In [ ]:
# 验证：prefill + 逐 token decode，结果应与一次性算全长一致
torch.manual_seed(0)
dim, heads = 64, 8
mha = LlamaMHA(dim, heads)
x_full = torch.randn(1, 5, dim)

# 一次性
out_full, _ = mha(x_full)

# prefill 前 3 + decode 后 2
out_pre, cache = mha(x_full[:, :3])
outs = [out_pre]
for t in range(3, 5):
    o, cache = mha(x_full[:, t:t+1], cache=cache)
    outs.append(o)
out_inc = torch.cat(outs, dim=1)
print('增量与全长一致:', torch.allclose(out_inc, out_full, atol=1e-5))
print('shape:', out_inc.shape)

## 小结 / 易错点
- KV cache 的 `cat` 在 `dim=2`（序列维），注意 head 维在 `dim=1`。
- decode 阶段 q 长度为 1，但 k/v 是全部历史，attention 输出仍是 `[B,h,1,d_k]`。
- 训练时不用 cache；推理 prefill 后接 decode 是标准流程。
- `attention_mask` 加法形式（$-\infty$ 填充）比乘法形式更稳。

## ✅ 测试验证

In [ ]:
# 验证 MHA + KV Cache
import torch

# 测试 LlamaMHA（假设已定义）
try:
    dim, num_heads = 32, 4
    mha = LlamaMHA(dim=dim, num_heads=num_heads)
    B, N = 2, 8
    x = torch.randn(B, N, dim)

    # 无 cache 前向
    out = mha(x)
    assert out.shape == (B, N, dim), f"shape wrong: {out.shape}"
    print("  ✓ 无 cache 前向形状正确:", out.shape)

    # KV Cache: prefill + decode
    # prefill: 处理前 N 个 token
    out_prefill = mha(x, use_cache=True)
    print("  ✓ Prefill 成功")

    # decode: 逐 token 生成
    x_new = torch.randn(B, 1, dim)
    out_decode = mha(x_new, use_cache=True)
    assert out_decode.shape == (B, 1, dim), f"decode shape wrong: {out_decode.shape}"
    print("  ✓ Decode 形状正确:", out_decode.shape)

except NameError:
    print("  (LlamaMHA 未定义，尝试基本 MHA 验证)")
    # 基本验证: attention 输出形状
    B, N, D, H = 2, 8, 32, 4
    x = torch.randn(B, N, D)
    q = k = v = x
    scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
    weights = torch.softmax(scores, dim=-1)
    out = torch.matmul(weights, v)
    assert out.shape == (B, N, D)
    print("  ✓ 基本 attention 形状正确:", out.shape)

print("✅ MHA 测试通过")
